# 04 — Strategy Simulation

Drives the Monte Carlo engine directly. Compares base vs what-if strategies, visualises position distributions, and shows the undercut threat model.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('../src')))
sys.path.insert(0, str(pathlib.Path('../apps/api')))
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Monte Carlo simulation inline (same logic as /whatif endpoint)
def simulate_strategy(driver_number, target_pit_lap, target_compound, push_delta,
                      remaining_laps, current_lap, n_sims=5000, seed=42):
    rng = np.random.default_rng(seed)
    compound_pace = {'SOFT': -0.4, 'MEDIUM': 0.0, 'HARD': 0.35, 'INTERMEDIATE': 0.8, 'WET': 1.4}
    pit_loss       = 22.0
    deg_per_lap    = {'SOFT': 0.06, 'MEDIUM': 0.025, 'HARD': 0.012}
    curr_compound  = 'MEDIUM'
    curr_age       = 15
    base_pace      = 85.0 + rng.normal(0, 0.1, n_sims)

    # Base: stay out
    base_time = np.zeros(n_sims)
    for lap in range(remaining_laps):
        age = curr_age + lap
        deg = deg_per_lap.get(curr_compound, 0.025) * age
        base_time += base_pace + deg + rng.normal(0, 0.3, n_sims)

    # What-if: pit at target_pit_lap
    whatif_time = np.zeros(n_sims)
    laps_before = max(0, target_pit_lap - current_lap)
    for lap in range(laps_before):
        age = curr_age + lap
        deg = deg_per_lap.get(curr_compound, 0.025) * age
        whatif_time += base_pace + deg + rng.normal(0, 0.3, n_sims)
    whatif_time += pit_loss
    new_pace = base_pace + compound_pace.get(target_compound, 0)
    for lap in range(remaining_laps - laps_before):
        deg = deg_per_lap.get(target_compound, 0.025) * lap + push_delta
        whatif_time += new_pace + deg + rng.normal(0, 0.3, n_sims)

    return base_time, whatif_time

base_t, whatif_t = simulate_strategy(
    driver_number=4, target_pit_lap=24, target_compound='HARD',
    push_delta=-0.2, remaining_laps=30, current_lap=20
)
print(f'Base strategy:   mean={base_t.mean():.1f}s  std={base_t.std():.1f}s')
print(f'What-if:         mean={whatif_t.mean():.1f}s  std={whatif_t.std():.1f}s')
print(f'Net delta:       {whatif_t.mean() - base_t.mean():+.2f}s')
print(f'Win probability: {(whatif_t < base_t).mean():.1%} in favour of pit')

In [ ]:
# Visualise the distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('#080c14')
for ax in (ax1, ax2):
    ax.set_facecolor('#0f172a')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
    ax.tick_params(colors='#8b9bb4')

delta = whatif_t - base_t
ax1.hist(base_t,   bins=60, alpha=0.7, color='#8b9bb4',  label='Base (stay out)',  density=True)
ax1.hist(whatif_t, bins=60, alpha=0.7, color='#22c55e',  label='What-if (pit lap 24 HARD)', density=True)
ax1.axvline(base_t.mean(),   color='white',   linewidth=1.5, linestyle='--')
ax1.axvline(whatif_t.mean(), color='#22c55e', linewidth=1.5, linestyle='--')
ax1.set_xlabel('Total race time (s)', color='#8b9bb4')
ax1.set_title('Strategy comparison — 5,000 samples', color='white', fontsize=11)
ax1.legend(facecolor='#0f172a', edgecolor='#1e293b', labelcolor='white', fontsize=8)

ax2.hist(delta, bins=60, color=np.where(delta.mean() < 0, '#22c55e', '#ff1801'), alpha=0.8)
ax2.axvline(0, color='white', linewidth=1.2, linestyle=':')
ax2.axvline(delta.mean(), color='#f59e0b', linewidth=1.5, linestyle='--', label=f'Mean Δ: {delta.mean():+.2f}s')
ax2.set_xlabel('Time delta (what-if − base, s)', color='#8b9bb4')
ax2.set_title(f'P(what-if faster): {(delta < 0).mean():.1%}', color='white', fontsize=11)
ax2.legend(facecolor='#0f172a', edgecolor='#1e293b', labelcolor='white', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Sweep: what pit lap is optimal?
pit_laps = range(20, 40)
current_lap = 18
remaining   = 45
deltas = []
for pit in pit_laps:
    bt, wt = simulate_strategy(4, pit, 'HARD', 0.0, remaining, current_lap, n_sims=2000)
    deltas.append(wt.mean() - bt.mean())

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#080c14'); ax.set_facecolor('#0f172a')
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
ax.tick_params(colors='#8b9bb4')
colors = ['#22c55e' if d < 0 else '#ff1801' for d in deltas]
ax.bar(list(pit_laps), deltas, color=colors, edgecolor='#1e293b', linewidth=0.5)
ax.axhline(0, color='white', linewidth=0.8)
best = list(pit_laps)[int(np.argmin(deltas))]
ax.axvline(best, color='#f59e0b', linewidth=1.5, linestyle='--', label=f'Optimal lap: {best}')
ax.set_xlabel('Target pit lap', color='#8b9bb4')
ax.set_ylabel('Mean time delta vs base (s)', color='#8b9bb4')
ax.set_title('Pit window sweep — HARD compound from lap 18', color='white', fontsize=12)
ax.legend(facecolor='#0f172a', edgecolor='#1e293b', labelcolor='white')
plt.tight_layout(); plt.show()
print(f'Optimal pit lap: {best}  (delta: {min(deltas):+.2f}s)')

In [ ]:
# Undercut threat model
import sys; sys.path.insert(0, '../src')
try:
    from pitwall.models.pit.opponent_model import OpponentPitModel
    model = OpponentPitModel(undercut_gap_threshold_s=1.8)
    # Norris (4) vs Verstappen (1): gap 1.24s, Norris on 21-lap MEDIUM, VER on 19-lap HARD
    threat = model.evaluate_undercut_threat(
        driver_number=4, driver_tyre_age=21, driver_compound='MEDIUM',
        rival_number=1,  rival_tyre_age=19,  rival_compound='HARD',
        gap_s=1.24
    )
    print('Undercut threat assessment:')
    for k, v in vars(threat).items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'Model not available in this env: {e}')

In [ ]:
# Safety car probability by circuit and lap
try:
    from pitwall.models.safety_car.hazard import SafetyCarHazardModel
    sc = SafetyCarHazardModel()
    circuits = ['monaco', 'monza', 'silverstone', 'bahrain']
    laps     = [1, 20, 50, 70]
    print(f'{"Circuit":<15} ' + '  '.join(f'Lap {l:2d}' for l in laps))
    print('-' * 60)
    for circuit in circuits:
        probs = []
        for lap in laps:
            h = sc.predict_hazard(circuit_id=circuit, lap_number=lap, total_laps=78)
            probs.append(h.p_sc_next_1)
        print(f'{circuit:<15} ' + '  '.join(f'{p:.1%}  ' for p in probs))
except Exception as e:
    print(f'Model not available: {e}')